<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Recurrent_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Vanilla RNN

In [ ]:
from datasets import load_dataset
from collections import Counter
import re


In [ ]:
dataset = load_dataset("glue", "sst2")

print(dataset)
print(dataset["train"][0])
print(dataset["train"].features)

train_data = dataset["train"].shuffle(seed=42).select(range(5000))
valid_data = dataset["validation"].shuffle(seed=42)

print("Train labels:", Counter(train_data["label"]))
print("Valid labels:", Counter(valid_data["label"]))

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b|[!?.,]", text.lower())


counter = Counter()

for item in train_data:
    tokens = tokenize(item["sentence"])
    counter.update(tokens)


vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

min_freq = 2

for word, count in counter.items():
    if count >= min_freq:
        vocab[word] = len(vocab)

print("Vocab size:", len(vocab))
print(list(vocab.items())[:20])

In [ ]:
max_length = 30

def encode_text(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    return ids


def pad_or_truncate(ids, max_length):
    if len(ids) > max_length:
        ids = ids[:max_length]

    return ids + [vocab["<PAD>"]] * (max_length - len(ids))

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, hf_dataset, vocab, max_length):
        self.dataset = hf_dataset
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["sentence"]
        label = self.dataset[idx]["label"]

        ids = encode_text(text, self.vocab)

        length = min(len(ids), self.max_length)

        ids = pad_or_truncate(ids, self.max_length)

        X = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        length = torch.tensor(length, dtype=torch.long)

        return X, y, length

In [ ]:
train_set = SentimentDataset(train_data, vocab, max_length)
valid_set = SentimentDataset(valid_data, vocab, max_length)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=64)

X_batch, y_batch, lengths_batch = next(iter(train_loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("lengths_batch shape:", lengths_batch.shape)
print("Example X:", X_batch[0])
print("Example y:", y_batch[0])
print("Example lengths:", lengths_batch[0])

In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, X, lengths):
        embedded = self.embedding(X)

        outputs, last_hidden = self.rnn(embedded)

        batch_size = outputs.size(0)

        final_hidden = outputs[
            torch.arange(batch_size, device=outputs.device),
            lengths - 1
        ]

        logits = self.fc(final_hidden)

        return logits

In [ ]:
vocab_size = len(vocab)
embedding_dim = 64
hidden_size = 128
num_classes = 2

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleRNNClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_classes=num_classes
).to(device)

X_batch, y_batch, lengths_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
lengths_batch = lengths_batch.to(device)

embedded = model.embedding(X_batch)
outputs, last_hidden = model.rnn(embedded)

batch_size = outputs.size(0)

# Correctly extract final_hidden using lengths_batch, as defined in SimpleRNNClassifier's forward method
final_hidden = outputs[
    torch.arange(batch_size, device=outputs.device),
    lengths_batch - 1
]

logits = model.fc(final_hidden)

print("X_batch shape:", X_batch.shape)
print("embedded shape:", embedded.shape)
print("outputs shape:", outputs.shape)
print("last_hidden shape:", last_hidden.shape)
print("final_hidden shape:", final_hidden.shape)
print("logits shape:", logits.shape)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    hidden_size=128,
    num_classes=2
).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch, lengths in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        lengths = lengths.to(device)

        logits = model(X_batch, lengths)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / total, correct / total

In [ ]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            lengths = lengths.to(device)

            logits = model(X_batch, lengths)
            loss = loss_fn(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / total, correct / total

In [ ]:
n_epochs = 10

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        valid_loader,
        loss_fn,
        device
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc:.4f} | "
        f"valid loss: {valid_loss:.4f} | "
        f"valid acc: {valid_acc:.4f}"
    )

In [ ]:
def predict_sentiment(text, model, vocab, max_length, device):
    model.eval()

    ids = encode_text(text, vocab)
    length = min(len(ids), max_length)
    ids = pad_or_truncate(ids, max_length)

    X = torch.tensor([ids], dtype=torch.long).to(device)
    lengths = torch.tensor([length], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(X, lengths)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label = "positive" if pred == 1 else "negative"

    return label, probs.cpu()

In [ ]:
sentences = [
    "this movie is amazing",
    "this movie is terrible",
    "i really enjoyed this film",
    "i did not like this movie",
    "a wonderful and emotional story",
    "boring and predictable"
]

for sentence in sentences:
    label, probs = predict_sentiment(
        sentence,
        model,
        vocab,
        max_length,
        device
    )

    print(sentence, "→", label, probs)

#Deep RNN

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

print(dataset)

In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])
print(dataset["train"][2])

In [ ]:
print(dataset["train"].features)

In [ ]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

dataset = dataset.shuffle(seed=42)

train_data = dataset["train"].shuffle(seed=42)
valid_data = dataset["validation"].shuffle(seed=42)
test_data = dataset["test"].shuffle(seed=42)

print("Train labels:", Counter(train_data["label"]))
print("Valid labels:", Counter(valid_data["label"]))
print("Test labels:", Counter(test_data["label"]))

In [ ]:
import re

def tokenize(text):
    return re.findall(r"\b\w+\b|[!?.,]", text.lower())


print(tokenize(train_data[0]["text"]))

In [ ]:
from collections import Counter

counter = Counter()

for item in train_data:
    tokens = tokenize(item["text"])
    counter.update(tokens)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

min_freq = 2

for word, count in counter.items():
    if count >= min_freq:
        vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

In [ ]:
max_length = 50

def encode_text(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    return ids


def pad_or_truncate(ids, max_length):
    if len(ids) > max_length:
        ids = ids[:max_length]

    padded = ids + [vocab["<PAD>"]] * (max_length - len(ids))
    return padded

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class RottenTomatoesDataset(Dataset):
    def __init__(self, hf_dataset, vocab, max_length):
        self.dataset = hf_dataset
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["text"]
        label = self.dataset[idx]["label"]

        ids = encode_text(text, self.vocab)

        length = min(len(ids), self.max_length)
        ids = pad_or_truncate(ids, self.max_length)

        X = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        length = torch.tensor(length, dtype=torch.long)

        return X, y, length

In [ ]:
train_set = RottenTomatoesDataset(train_data, vocab, max_length)
valid_set = RottenTomatoesDataset(valid_data, vocab, max_length)
test_set = RottenTomatoesDataset(test_data, vocab, max_length)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)

In [ ]:
X_batch, y_batch, lengths_batch = next(iter(train_loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("lengths_batch shape:", lengths_batch.shape)
print("X_batch[0]:", X_batch[0])
print("y_batch[0]:", y_batch[0])
print("lengths_batch[0]:", lengths_batch[0])

In [ ]:
import torch
import torch.nn as nn


class DeepBiRNNTextClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_size,
        num_classes,
        num_layers,
        dropout
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, X, lengths):
        embedded = self.embedding(X)

        outputs, last_hidden = self.rnn(embedded)

        batch_size = outputs.size(0)
        hidden_twice = outputs.size(2)
        hidden_size = hidden_twice // 2

        forward_last = outputs[
            torch.arange(batch_size, device=outputs.device),
            lengths - 1,
            :hidden_size
        ]

        backward_last = outputs[
            torch.arange(batch_size, device=outputs.device),
            0,
            hidden_size:
        ]

        final_hidden = torch.cat(
            [forward_last, backward_last],
            dim=1
        )

        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)

        return logits

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)

model = DeepBiRNNTextClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_size=128,
    num_classes=2,
    num_layers=2,
    dropout=0.3
).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:
X_batch, y_batch, lengths_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
lengths_batch = lengths_batch.to(device)

embedded = model.embedding(X_batch)
outputs, last_hidden = model.rnn(embedded)

batch_size = outputs.size(0)
hidden_twice = outputs.size(2)
hidden_size = hidden_twice // 2

forward_last = outputs[
    torch.arange(batch_size, device=outputs.device),
    lengths_batch - 1,
    :hidden_size
]

backward_last = outputs[
    torch.arange(batch_size, device=outputs.device),
    0,
    hidden_size:
]

final_hidden = torch.cat(
    [forward_last, backward_last],
    dim=1
)

logits = model.fc(final_hidden)

print("X_batch shape:", X_batch.shape)
print("embedded shape:", embedded.shape)
print("outputs shape:", outputs.shape)
print("last_hidden shape:", last_hidden.shape)
print("forward_last shape:", forward_last.shape)
print("backward_last shape:", backward_last.shape)
print("final_hidden shape:", final_hidden.shape)
print("logits shape:", logits.shape)

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch, lengths_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        lengths_batch = lengths_batch.to(device)

        logits = model(X_batch, lengths_batch)

        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            lengths_batch = lengths_batch.to(device)

            logits = model(X_batch, lengths_batch)

            loss = loss_fn(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
import copy

n_epochs = 15

best_valid_acc = 0
best_model_state = None

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        valid_loader,
        loss_fn,
        device
    )

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc:.4f} | "
        f"valid loss: {valid_loss:.4f} | "
        f"valid acc: {valid_acc:.4f}"
    )

print("Best valid accuracy:", best_valid_acc)

model.load_state_dict(best_model_state)

In [ ]:
test_loss, test_acc = evaluate(
    model,
    test_loader,
    loss_fn,
    device
)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

In [ ]:
def predict_sentiment(text, model, vocab, max_length, device):
    model.eval()

    ids = encode_text(text, vocab)

    length = min(len(ids), max_length)

    ids = pad_or_truncate(ids, max_length)

    X = torch.tensor([ids], dtype=torch.long).to(device)
    lengths = torch.tensor([length], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(X, lengths)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label = "positive" if pred == 1 else "negative"

    return label, probs.cpu()

In [ ]:
sentences = [
    "this movie was amazing",
    "this movie was terrible",
    "i really enjoyed this film",
    "i did not like this movie",
    "the film was boring and predictable",
    "the story was wonderful and emotional"
]

for sentence in sentences:
    label, probs = predict_sentiment(
        sentence,
        model,
        vocab,
        max_length,
        device
    )

    print(sentence, "→", label, probs)